<a href="https://colab.research.google.com/github/RajPShinde/MLP-Neural-Nets/blob/main/MLP_Tabular_Classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
df = pd.read_csv(url)
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [2]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB


In [3]:
df_clean = df.copy()

df_clean['Age'] = df_clean['Age'].fillna(df_clean['Age'].median())
df_clean['Embarked'] = df_clean['Embarked'].fillna(df_clean['Embarked'].mode()[0])

df_clean['Deck'] = df_clean['Cabin'].str[0].fillna('Unknown')

df_clean = df_clean.drop(columns=['Cabin', 'Name', 'Ticket', 'PassengerId'])

df_encoded = pd.get_dummies(df_clean, columns=['Sex', 'Embarked', 'Deck'])

df_encoded.head()

,Survived,Pclass,Age,SibSp,Parch,Fare,Sex_female,Sex_male,Embarked_C,Embarked_Q,Embarked_S,Deck_A,Deck_B,Deck_C,Deck_D,Deck_E,Deck_F,Deck_G,Deck_T,Deck_Unknown
0,0,3,22.0,1,0,7.2500,False,True,False,False,True,False,False,False,False,False,False,False,False,True
1,1,1,38.0,1,0,71.2833,True,False,True,False,False,False,False,True,False,False,False,False,False,False
2,1,3,26.0,0,0,7.9250,True,False,False,False,True,False,False,False,False,False,False,False,False,True
3,1,1,35.0,1,0,53.1000,True,False,False,False,True,False,False,True,False,False,False,False,False,False
4,0,3,35.0,0,0,8.0500,False,True,False,False,True,False,False,False,False,False,False,False,False,True


In [4]:
from sklearn.model_selection import train_test_split

X = df_encoded.drop(columns=['Survived']).values.astype('float32')
y = df_encoded['Survived'].values.astype('int64')

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

print(X_train.shape, X_val.shape, X_test.shape)

(623, 19) (134, 19) (134, 19)


In [5]:
import numpy as np

baseline_acc = max(y_test.mean(), 1 - y_test.mean())
print(f'Baseline (always guess majority class) accuracy: {baseline_acc:.3f}')

Baseline (always guess majority class) accuracy: 0.619


In [6]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train).astype('float32')
X_val_s = scaler.transform(X_val).astype('float32')
X_test_s = scaler.transform(X_test).astype('float32')

In [7]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

train_ds = TensorDataset(torch.tensor(X_train_s), torch.tensor(y_train))
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)

X_val_t = torch.tensor(X_val_s).to(device)
y_val_t = torch.tensor(y_val).to(device)
X_test_t = torch.tensor(X_test_s).to(device)
y_test_t = torch.tensor(y_test).to(device)

cuda


In [9]:
class TitanicNet(nn.Module):
  def __init__(self, input_dim):
    super().__init__()
    self.net = nn.Sequential(
      nn.Linear(input_dim, 32),
      nn.ReLU(),
      nn.Linear(32, 16),
      nn.ReLU(),
      nn.Linear(16, 2)
  )
  def forward(self, x):
      return self.net(x)

In [10]:
model = TitanicNet(X_train_s.shape[1]).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

n_epochs = 50
for epoch in range(n_epochs):
  model.train()
  for xb, yb in train_loader:
    xb, yb = xb.to(device), yb.to(device)
    optimizer.zero_grad()
    logits = model(xb)
    loss = criterion(logits, yb)
    loss.backward()
    optimizer.step()
  if (epoch + 1) % 10 == 0:
    model.eval()
    with torch.no_grad():
      val_preds = model(X_val_t).argmax(dim=1)
      val_acc = (val_preds == y_val_t).float().mean().item()
      print(f'Epoch {epoch+1}: loss={loss.item():.3f}, val_acc={val_acc:.3f}')

Epoch 10: loss=0.495, val_acc=0.828
Epoch 20: loss=0.180, val_acc=0.836
Epoch 30: loss=0.780, val_acc=0.858
Epoch 40: loss=0.274, val_acc=0.858
Epoch 50: loss=0.400, val_acc=0.866


In [11]:
model.eval()
with torch.no_grad():
  test_preds = model(X_test_t).argmax(dim=1)
  test_acc = (test_preds == y_test_t).float().mean().item()
print(f'Baseline accuracy: {baseline_acc:.3f}')
print(f'Final TEST accuracy: {test_acc:.3f}')

Baseline accuracy: 0.619
Final TEST accuracy: 0.813
